# Sales Forecasting & Customer Segmentation (SQL)
Runs on Google Colab using **DuckDB** (SQL engine, no server needed).

**Steps:** 1) Install → 2) Load data → 3) Create `sales` table → 4) Run SQL analysis → 5) Charts

**Dataset:** Superstore Sales CSV from Kaggle. Or set `USE_SYNTHETIC = True` below to try it with generated data.

In [ ]:
!pip install duckdb -q

In [ ]:
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt

# True  -> generated sample data (no download needed)
# False -> upload the Superstore CSV from Kaggle when prompted
USE_SYNTHETIC = False

if USE_SYNTHETIC:
    rng = np.random.default_rng(42)
    n = 10000
    dates = pd.to_datetime("2014-01-01") + pd.to_timedelta(rng.integers(0, 4*365, n), unit="D")
    t = (dates - dates.min()).days / 30
    season = 1 + 0.35*np.sin((dates.month - 3) / 12 * 2*np.pi) + 0.25*(dates.month >= 11)
    sales = np.round(rng.gamma(2, 60, n) * season * (1 + t/60), 2)
    df = pd.DataFrame({
        "order_id": [f"ORD-{i//2}" for i in range(n)],
        "order_date": dates,
        "customer_id": [f"C{rng.integers(1, 800):04d}" for _ in range(n)],
        "category": rng.choice(["Furniture", "Technology", "Office Supplies"], n),
        "region": rng.choice(["East", "West", "Central", "South"], n),
        "sales": sales,
        "quantity": rng.integers(1, 10, n),
        "profit": np.round(sales * rng.normal(0.12, 0.15, n), 2),
    })
else:
    from google.colab import files
    up = files.upload()
    raw = pd.read_csv(list(up.keys())[0], encoding="latin-1")
    raw.columns = [c.strip().lower().replace(" ", "_").replace("-", "_") for c in raw.columns]
    df = raw[["order_id", "order_date", "customer_id", "category", "region", "sales", "quantity", "profit"]].copy()
    # If dates look wrong, switch to dayfirst=True
    df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=False)

print(df.shape)
df.head()

## 1. Create the `sales` table

In [ ]:
con = duckdb.connect()
con.register("df_in", df)
con.execute("CREATE TABLE sales AS SELECT * FROM df_in")
con.execute("ALTER TABLE sales ALTER order_date TYPE DATE")

def q(sql):
    return con.execute(sql).df()

q("SELECT COUNT(*) AS rows, MIN(order_date) AS first_order, MAX(order_date) AS last_order FROM sales")

## 2. Monthly aggregation

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW monthly_sales AS
SELECT CAST(DATE_TRUNC('month', order_date) AS DATE) AS month,
       SUM(sales)               AS revenue,
       SUM(profit)              AS profit,
       COUNT(DISTINCT order_id) AS orders
FROM sales
GROUP BY 1
""")
q("SELECT * FROM monthly_sales ORDER BY month LIMIT 12")

## 3. Moving average, MoM and YoY growth

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW sales_trends AS
SELECT month,
       revenue,
       ROUND(AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS moving_avg_3m,
       ROUND(100.0 * (revenue - LAG(revenue, 1)  OVER (ORDER BY month)) / NULLIF(LAG(revenue, 1)  OVER (ORDER BY month), 0), 2) AS mom_growth_pct,
       ROUND(100.0 * (revenue - LAG(revenue, 12) OVER (ORDER BY month)) / NULLIF(LAG(revenue, 12) OVER (ORDER BY month), 0), 2) AS yoy_growth_pct
FROM monthly_sales
""")
trends = q("SELECT * FROM sales_trends ORDER BY month")
trends.tail(12)

## 4. 3-month forecast (linear trend × seasonal index)

In [ ]:
forecast = q("""
WITH idx AS (
    SELECT month, revenue, ROW_NUMBER() OVER (ORDER BY month) AS t
    FROM monthly_sales
),
fit AS (
    SELECT REGR_SLOPE(revenue, t)     AS slope,
           REGR_INTERCEPT(revenue, t) AS intercept,
           MAX(t)                     AS last_t,
           MAX(month)                 AS last_month
    FROM idx
),
seasonal AS (
    SELECT EXTRACT(MONTH FROM i.month)::INT AS m,
           AVG(i.revenue / (f.intercept + f.slope * i.t)) AS seasonal_index
    FROM idx i CROSS JOIN fit f
    GROUP BY 1
)
SELECT CAST(f.last_month + INTERVAL (g.h) MONTH AS DATE) AS forecast_month,
       ROUND((f.intercept + f.slope * (f.last_t + g.h)) * s.seasonal_index, 2) AS forecast_revenue
FROM fit f
CROSS JOIN generate_series(1, 3) AS g(h)
JOIN seasonal s
  ON s.m = EXTRACT(MONTH FROM CAST(f.last_month + INTERVAL (g.h) MONTH AS DATE))::INT
ORDER BY forecast_month
""")
forecast

## 5. Backtest: train on all but the last 6 months, measure MAPE

In [ ]:
mape = q("""
WITH idx AS (
    SELECT month, revenue, ROW_NUMBER() OVER (ORDER BY month) AS t
    FROM monthly_sales
),
cutoff AS (SELECT MAX(t) - 6 AS c FROM idx),
fit AS (
    SELECT REGR_SLOPE(i.revenue, i.t)     AS slope,
           REGR_INTERCEPT(i.revenue, i.t) AS intercept
    FROM idx i, cutoff co
    WHERE i.t <= co.c
),
seasonal AS (
    SELECT EXTRACT(MONTH FROM i.month)::INT AS m,
           AVG(i.revenue / (f.intercept + f.slope * i.t)) AS seasonal_index
    FROM idx i, cutoff co, fit f
    WHERE i.t <= co.c
    GROUP BY 1
)
SELECT ROUND(AVG(ABS(x.revenue - x.predicted) / x.revenue) * 100, 2) AS mape_pct
FROM (
    SELECT i.month, i.revenue,
           (f.intercept + f.slope * i.t) * s.seasonal_index AS predicted
    FROM idx i
    CROSS JOIN cutoff co
    CROSS JOIN fit f
    JOIN seasonal s ON s.m = EXTRACT(MONTH FROM i.month)::INT
    WHERE i.t > co.c
) x
""")
print("Backtest MAPE (%):", mape.iloc[0, 0])
mape

## 6. RFM customer segmentation

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW customer_rfm AS
WITH rfm AS (
    SELECT customer_id,
           (SELECT MAX(order_date) FROM sales) - MAX(order_date) AS recency_days,
           COUNT(DISTINCT order_id) AS frequency,
           SUM(sales)               AS monetary
    FROM sales
    GROUP BY customer_id
),
scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
           NTILE(5) OVER (ORDER BY frequency)         AS f_score,
           NTILE(5) OVER (ORDER BY monetary)          AS m_score
    FROM rfm
)
SELECT customer_id, recency_days, frequency, ROUND(monetary, 2) AS monetary,
       r_score, f_score, m_score,
       CASE
           WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
           WHEN r_score >= 3 AND f_score >= 3                  THEN 'Loyal Customers'
           WHEN r_score >= 4 AND f_score <= 2                  THEN 'New Customers'
           WHEN r_score <= 2 AND f_score >= 4                  THEN 'At Risk'
           WHEN r_score <= 2 AND f_score <= 2                  THEN 'Lost'
           ELSE 'Needs Attention'
       END AS segment
FROM scored
""")

segments = q("""
SELECT segment,
       COUNT(*)                AS customers,
       ROUND(SUM(monetary), 2) AS revenue,
       ROUND(100.0 * SUM(monetary) / SUM(SUM(monetary)) OVER (), 2) AS revenue_share_pct
FROM customer_rfm
GROUP BY segment
ORDER BY revenue DESC
""")
segments

## 7. Charts (screenshot these for your README)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

# Actual vs moving average vs forecast
ax[0].plot(trends["month"], trends["revenue"], label="Actual revenue", marker="o", markersize=3)
ax[0].plot(trends["month"], trends["moving_avg_3m"], label="3-month moving avg", linestyle="--")
ax[0].plot(forecast["forecast_month"], forecast["forecast_revenue"], label="Forecast", marker="s", color="red")
ax[0].set_title("Monthly Revenue & 3-Month Forecast")
ax[0].legend()
ax[0].tick_params(axis="x", rotation=45)

# Revenue share by RFM segment
ax[1].bar(segments["segment"], segments["revenue_share_pct"], color="teal")
ax[1].set_title("Revenue Share by RFM Segment (%)")
ax[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("sales_forecast_charts.png", dpi=150)
plt.show()

## Results to put on your resume
- Backtest MAPE: *(from section 5)*
- 3-month forecast: *(from section 4)*
- Top segment's revenue share: *(from section 6)*

To download the chart: `from google.colab import files; files.download("sales_forecast_charts.png")`